In [ ]:
### this code for generating apps.json file from apps.txt file

import json

# Open the file and read the data
with open('apps.txt', 'r', encoding='utf-8') as file:
    data = file.read()

# split the data into lines
lines = data.splitlines()

# get the header and remove empty strings resulted from split
header = [h for h in lines[0].split("  ") if h]

# process each line
out = []
for line in lines[2:]:
    # split each line on two or more spaces and remove empty strings resulted from split
    items = [i for i in line.split("  ") if i]

    # Ensure there are at least 2 items (Name and Id) before processing
    if len(items) >= 2:
        # map the Id to the output list
        out.append({"Id": items[1]})

# output to json
json_data = json.dumps(out, indent=4)

# print json_data
print(json_data)

# Write the json data to a file
with open('apps.json', 'w') as file:
    file.write(json_data)


In [1]:
### for removing duplicate in apps.json

import json
import time

# Start timing
start_time = time.time()

# Step 1: Read the JSON file
try:
    with open('apps.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
except Exception as e:
    print(f"Error reading the file: {e}")
    exit()

# Step 2: Remove duplicates based on the "Id" field
unique_data = []
seen_ids = set()

for item in data:
    trimmed_id = item['Id'].strip()  # Remove leading/trailing spaces from the "Id"
    if trimmed_id not in seen_ids:
        seen_ids.add(trimmed_id)  # Mark this Id as seen
        unique_data.append(item)  # Add the item to the result array

# Step 3: Write the unique data back to the file
try:
    with open('apps.json', 'w', encoding='utf-8') as file:
        json.dump(unique_data, file, ensure_ascii=False, indent=2)
    print('File updated successfully with unique entries.')
except Exception as e:
    print(f"Error writing to the file: {e}")
    exit()

# End timing and print the result
end_time = time.time()
execution_time = end_time - start_time
print(f"Execution Time: {execution_time:.4f} seconds")


File updated successfully with unique entries.
Execution Time: 0.0170 seconds


In [ ]:
### for fetching app details using winget show

import json
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm  # For the progress bar

# Input and output file paths
input_file = "apps.json"
output_file = "winget_results.json"

# Load app IDs from the JSON file
with open(input_file, "r", encoding="utf-8") as f:
    app_list = json.load(f)



# Function to fetch app details using `winget show`
def fetch_app_details(app):
    app_id = app["Id"].strip()  # Remove any leading/trailing whitespace

    try:
        # Run the `winget show` command
        winget_output = subprocess.check_output(
            ["winget", "show", app_id],
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8"  # Use UTF-8 encoding
        )

        # Parse the output
        parsed_result = {
            "Id": app_id,
            "Name": "",
            "Version": "",
            "Publisher": "",
            "PublisherUrl": "",
            "Description": "",
            "Homepage": "",
            "License": "",
            "Installer Url": ""
        }

        # Extract details from the winget output
        for line in winget_output.splitlines():
            line = line.strip()
            if line.startswith("Found"):
                parsed_result["Name"] = line.split("Found ")[1].split("[")[0].strip()
            elif line.startswith("Version:"):
                parsed_result["Version"] = line.split("Version:")[1].strip()
            elif line.startswith("Publisher:"):
                parsed_result["Publisher"] = line.split("Publisher:")[1].strip()
            elif line.startswith("Publisher Url:"):
                parsed_result["PublisherUrl"] = line.split("Publisher Url:")[1].strip()
            elif line.startswith("Description:"):
                parsed_result["Description"] = line.split("Description:")[1].strip()
            elif line.startswith("Homepage:"):
                parsed_result["Homepage"] = line.split("Homepage:")[1].strip()
            elif line.startswith("License:"):
                parsed_result["License"] = line.split("License:")[1].strip()
            elif line.startswith("Installer Url:"):
                parsed_result["Installer Url"] = line.split("Installer Url:")[1].strip()

        return parsed_result

    except (subprocess.CalledProcessError, UnicodeDecodeError):
        return None


results = []
with ThreadPoolExecutor(max_workers=5) as executor:  # Adjust the number of workers as needed
    future_to_app = {executor.submit(fetch_app_details, app): app for app in app_list}
    for future in tqdm(as_completed(future_to_app), total=len(future_to_app), desc="Processing Apps"):
        result = future.result()
        if result:
            results.append(result)

# Save the results to the output JSON file
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"Results saved to {output_file}")

In [1]:
### this code for fetching winget_results.json file to a mongodb database

import json
import motor.motor_asyncio

# Load the JSON data from the file
with open('winget_results.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Connect to the MongoDB server
client = motor.motor_asyncio.AsyncIOMotorClient('mongodb://localhost:27017')

# Access the "winget" database
db = client['WinStore']

# Access the "apps" collection

collection = db['apps']

# Insert the data into the collection
result = await collection.insert_many(data)

